In [0]:
# ===========================================
# Notebook Name:
# 04_validate_v2_migration
#
# Purpose:
# Validate that the v2 migration (Step1-4) left
# the catalog in a consistent, usable state before
# Ingest/Silver/Gold v2 notebooks (Step5-7) start
# writing to these tables.
#
# Checks:
# - migration_inventory has no REBUILD rows for
#   tables that were supposed to already exist
# - primary-key uniqueness on key tables
# - required columns are not null
# - deck_hash reproducibility (ADR-001)
#
# Any failed check displays the offending rows
# and raises, stopping downstream execution.
# ===========================================

In [0]:
from pyspark.sql import functions as F

MIGRATION_INVENTORY_TABLE = "pokemon.ops.migration_inventory"

validation_failures = []

In [0]:
# -------------------------------------------
# Check 1: v2 target tables exist (no REBUILD
# rows left for tables this notebook expects to
# be usable downstream).
# -------------------------------------------
EXPECTED_TO_EXIST_BY_NOW = {
    "bronze.tournament_list_raw",
    "bronze.event_result_raw",
    "bronze.deck_raw",
    "bronze.scrape_log",
    "silver.tournaments",
    "silver.tournament_results",
    "silver.decks",
    "silver.deck_cards",
    "gold.card_usage",
    "gold.deck_registry",
    "gold.deck_pokemon_features",
    "gold.deck_similarity",
    "gold.deck_archetypes",
    "ops.result_fetch_targets",
    "ops.deck_fetch_targets",
    "ops.pipeline_run_log",
    "ops.pipeline_state",
    "ops.migration_inventory",
}

rebuild_rows_df = (
    spark.table(MIGRATION_INVENTORY_TABLE)
    .filter(F.col("v2_action") == "REBUILD")
    .select("schema", "table_name")
    .distinct()
)

rebuild_keys = {
    f"{row['schema']}.{row['table_name']}"
    for row in rebuild_rows_df.collect()
}

still_missing = EXPECTED_TO_EXIST_BY_NOW & rebuild_keys

if still_missing:
    print("Tables still marked REBUILD:", sorted(still_missing))
    validation_failures.append(
        f"{len(still_missing)} v2 tables not yet created: {sorted(still_missing)}"
    )
else:
    print("Check 1 passed: no expected v2 tables are missing.")

In [0]:
# -------------------------------------------
# Check 2: primary-key uniqueness on key tables.
# -------------------------------------------
PK_CHECKS = [
    ("pokemon.silver.tournaments", ["tournament_id"]),
    ("pokemon.silver.decks", ["deck_hash"]),
    ("pokemon.gold.deck_registry", ["deck_hash"]),
    ("pokemon.gold.deck_similarity", ["deck_hash_a", "deck_hash_b"]),
    ("pokemon.ops.result_fetch_targets", ["tournament_id"]),
]

for table_name, pk_columns in PK_CHECKS:
    dup_df = (
        spark.table(table_name)
        .groupBy(*pk_columns)
        .count()
        .filter(F.col("count") > 1)
    )

    dup_count = dup_df.count()

    if dup_count > 0:
        print(f"{table_name}: {dup_count} duplicate key(s) on {pk_columns}")
        display(dup_df)
        validation_failures.append(
            f"{table_name}: {dup_count} duplicate primary keys on {pk_columns}"
        )
    else:
        print(f"{table_name}: primary key {pk_columns} is unique.")

In [0]:
# -------------------------------------------
# Check 3: required columns are not null.
# -------------------------------------------
NOT_NULL_CHECKS = [
    ("pokemon.silver.tournaments", ["tournament_id", "event_date"]),
    ("pokemon.silver.tournament_results", ["tournament_id", "rank"]),
    ("pokemon.silver.decks", ["deck_hash", "deck_code"]),
    ("pokemon.silver.deck_cards", ["deck_hash", "card_name", "quantity"]),
]

for table_name, required_columns in NOT_NULL_CHECKS:
    table_df = spark.table(table_name)

    for column_name in required_columns:
        null_count = table_df.filter(
            F.col(column_name).isNull()
        ).count()

        if null_count > 0:
            print(f"{table_name}.{column_name}: {null_count} null value(s)")
            validation_failures.append(
                f"{table_name}.{column_name}: {null_count} unexpected nulls"
            )
        else:
            print(f"{table_name}.{column_name}: no nulls.")

In [0]:
# -------------------------------------------
# Check 4: deck_hash reproducibility (ADR-001).
#
# Recompute deck_hash from silver.deck_cards using
# the same normalization as the Silver builder
# (sorted, aggregated card_name|quantity lines,
# SHA-256), and compare against silver.decks.deck_hash.
# -------------------------------------------
recomputed_hash_df = (
    spark.table("pokemon.silver.deck_cards")
    .groupBy("deck_hash", "card_name")
    .agg(F.sum("quantity").alias("quantity"))
    .groupBy("deck_hash")
    .agg(
        # Sort by (card_name, quantity) struct so ties are
        # impossible (card_name is unique after aggregation)
        # and prefix names (e.g. "Pikachu" vs "Pikachu VMAX")
        # sort the same way build_deck_hash's sorted() does.
        F.array_sort(
            F.collect_list(
                F.struct("card_name", "quantity")
            )
        ).alias("sorted_cards")
    )
    .withColumn(
        "canonical_string",
        F.concat_ws(
            "\n",
            F.transform(
                F.col("sorted_cards"),
                lambda c: F.concat_ws(
                    "|", c["card_name"], c["quantity"]
                ),
            ),
        ),
    )
    .withColumn(
        "recomputed_deck_hash",
        F.sha2(F.col("canonical_string"), 256),
    )
    .select("deck_hash", "recomputed_deck_hash")
)

hash_mismatch_df = (
    recomputed_hash_df
    .filter(F.col("deck_hash") != F.col("recomputed_deck_hash"))
)

hash_mismatch_count = hash_mismatch_df.count()

if hash_mismatch_count > 0:
    print(f"deck_hash mismatches: {hash_mismatch_count}")
    display(hash_mismatch_df)
    validation_failures.append(
        f"{hash_mismatch_count} deck_hash values do not reproduce per ADR-001"
    )
else:
    print("Check 4 passed: all deck_hash values reproduce per ADR-001.")

In [ ]:
# -------------------------------------------
# Check 5: legacy tournaments column backfill
# completeness, ahead of dropping the 14 legacy
# silver.tournaments columns.
#
# Of the 14 legacy columns slated for removal,
# only 4 have a v2 counterpart populated by a
# COALESCE backfill in 03_migrate_existing_data:
#   venue              -> venue_name
#   prefecture_name    -> prefecture
#   event_type_title   -> event_type
#   event_date_display -> event_date
#
# The other 10 legacy columns (event_title,
# event_type_id, address, shop_name, league,
# regulation, capacity, result_count, source_url,
# response_hash) have no v2 counterpart at all;
# they are dropped outright because nothing in
# Gold/ML/Analysis/src reads them (confirmed by
# repo-wide search), so there is nothing to
# backfill-verify for those 10.
#
# This check fails if any row has a legacy value
# but a NULL v2 counterpart, which would mean the
# backfill in 03_migrate_existing_data has not
# been (re)run since new legacy data arrived.
# -------------------------------------------
BACKFILL_PAIRS = [
    ("venue", "venue_name"),
    ("prefecture_name", "prefecture"),
    ("event_type_title", "event_type"),
    ("event_date_display", "event_date"),
]

tournaments_df = spark.table("pokemon.silver.tournaments")

for legacy_column, v2_column in BACKFILL_PAIRS:
    unbackfilled_df = tournaments_df.filter(
        F.col(legacy_column).isNotNull()
        & F.col(v2_column).isNull()
    )

    unbackfilled_count = unbackfilled_df.count()

    if unbackfilled_count > 0:
        print(
            f"{legacy_column} -> {v2_column}: "
            f"{unbackfilled_count} row(s) with legacy "
            "value but NULL v2 counterpart"
        )
        display(
            unbackfilled_df.select(
                "tournament_id", legacy_column, v2_column
            )
        )
        validation_failures.append(
            f"silver.tournaments: {unbackfilled_count} row(s) "
            f"have {legacy_column} set but {v2_column} is NULL "
            "(backfill incomplete)"
        )
    else:
        print(
            f"{legacy_column} -> {v2_column}: "
            f"backfill complete, safe to drop {legacy_column}."
        )

print(
    "Check 5 note: the remaining 10 legacy columns "
    "(event_title, event_type_id, address, shop_name, "
    "league, regulation, capacity, result_count, "
    "source_url, response_hash) have no v2 counterpart "
    "and are not backfill-checked; they are dropped "
    "because they are unread downstream."
)

In [0]:
# -------------------------------------------
# Final verdict.
# -------------------------------------------
if validation_failures:
    for failure in validation_failures:
        print("FAILED:", failure)

    raise ValueError(
        f"{len(validation_failures)} v2 migration validation "
        f"check(s) failed. See output above."
    )

print("All v2 migration validation checks passed.")